In [1]:
from os import walk
import pydicom as dicom
import numpy as np


images = []
labels = []
for _, dirnames, _ in walk('data'):
	for dir in dirnames:
		for dirpath, dirnames, filenames in walk('data/' + dir):
			for file_name in filenames:
				file = dirpath + '/' + file_name
				pneumothorax = 0
				if dir == 'pneumotorax_anon':
					pneumothorax = np.array([dicom.dcmread(file).pixel_array ])[0]
					for pneumo in pneumothorax:
						image = pneumo
						image = (image - np.min(image)) / (np.max(image) - np.min(image))
						image = (image * 255).astype(np.uint8)
						images.append(image)
				else:
					image = np.array(dicom.dcmread(file).pixel_array)
					image = (image - np.min(image)) / (np.max(image) - np.min(image))
					image = (image * 255).astype(np.uint8)
					images.append(image)
			if dir == 'norma_anon':
				labels += [1] * len(filenames)
			else:
				labels += [0] * len(filenames) if len(filenames) > 1 else [1] * len(pneumothorax)
images = np.array(images)
labels = np.array(labels)


<h3>Кластеризация</h3>

In [3]:
import torch
import torch.nn as nn
from torch.nn.functional import relu
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib


class CNNFeatureExtractor(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        
        x = self.conv1(x)
        x = relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = relu(x)
        x = self.pool(x)
        
        x = self.conv3(x)
        x = relu(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)               
        return x


class CNN_KNN_Model:

	def __init__(self, weights_path: str = None):
		self.device = "cuda" if torch.cuda.is_available() else "cpu"
		self.cnn = CNNFeatureExtractor().to(self.device)
		if weights_path is not None:
			self.load(weights_path)
		else:
			self.knn = None
			self.scaler = StandardScaler()

	def extract_features(self, images: np.ndarray, batch_size=16) -> np.ndarray:
		if images.ndim == 2:
			images = np.expand_dims(images, axis=0)

		self.cnn.eval()
		dataset = TensorDataset(torch.tensor(images, dtype=torch.float32).unsqueeze(1))
		loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

		all_features = []
		with torch.no_grad():
			for batch in loader:
				x = batch[0].to(self.device)
				feats = self.cnn(x)
				all_features.append(feats.cpu().numpy())
		return np.vstack(all_features)

	def fit(self, X: np.ndarray, cluster_range=(2, 15)):
		features = self.extract_features(X)
		features = self.scaler.fit_transform(features)

		best_score = -1
		best_k = None
		best_model = None

		for k in range(cluster_range[0], cluster_range[1] + 1):
			model = KMeans(n_clusters=k, random_state=42, n_init=10)
			labels = model.fit_predict(features)
			score = silhouette_score(features, labels)
			print(f"k={k}, silhouette={score:.4f}")
			if score > best_score:
				best_score = score
				best_k = k
				best_model = model

		self.knn = best_model
		print(f"Выбран k={best_k} с silhouette={best_score:.4f}")

	def predict(self, X: np.ndarray) -> np.ndarray:
		"""
		Предсказание классов
		"""
		if self.knn is None:
			raise ValueError("Модель KNN не обучена. Сначала вызовите fit().")

		features = self.extract_features(X)
		features = self.scaler.transform(features)
		return self.knn.predict(features)

	def save(self, path: str='model.pkl'):
		"""
		Сохраняем KNN и StandardScaler
		"""
		if self.knn is None:
			raise ValueError("Нет обученной модели KNN для сохранения.")
		joblib.dump({"knn": self.knn, "scaler": self.scaler}, path)

	def load(self, path: str='model.pkl'):
		"""
		Загружаем KNN и StandardScaler
		"""
		checkpoint = joblib.load(path)
		self.knn = checkpoint["knn"]
		self.scaler = checkpoint["scaler"]


In [23]:
model = CNN_KNN_Model()
model.fit(images)

k=2, silhouette=0.6297
k=3, silhouette=0.4586
k=4, silhouette=0.4482
k=5, silhouette=0.4671
k=6, silhouette=0.4749
k=7, silhouette=0.4769
k=8, silhouette=0.4758
k=9, silhouette=0.4746
k=10, silhouette=0.2145
k=11, silhouette=0.2186
k=12, silhouette=0.2096
k=13, silhouette=0.2107
k=14, silhouette=0.2135
k=15, silhouette=0.2110
Выбран k=2 с silhouette=0.6297


In [27]:
model.save('model.pkl')

In [33]:
image = images[0]
new_model = CNN_KNN_Model('model.pkl')
print(new_model.predict(image))

[0]


<h3>Классификация</h3>

In [9]:
import torch
import torch.nn as nn
from torch.nn.functional import relu
from torch.utils.data import DataLoader, TensorDataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib


class CNNFeatureExtractor(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        
        x = self.conv1(x)
        x = relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = relu(x)
        x = self.pool(x)
        
        x = self.conv3(x)
        x = relu(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)               
        return x


class CNN_KNN_Model:

	def __init__(self, weights_path: str = None):
		self.device = "cuda" if torch.cuda.is_available() else "cpu"
		self.cnn = CNNFeatureExtractor().to(self.device)
		if weights_path is not None:
			self.load(weights_path)
		else:
			self.knn = None
			self.scaler = StandardScaler()

	def extract_features(self, images: np.ndarray, batch_size=16) -> np.ndarray:
		if images.ndim == 2:
			images = np.expand_dims(images, axis=0)

		self.cnn.eval()
		dataset = TensorDataset(torch.tensor(images, dtype=torch.float32).unsqueeze(1))
		loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

		all_features = []
		with torch.no_grad():
			for batch in loader:
				x = batch[0].to(self.device)
				feats = self.cnn(x)
				all_features.append(feats.cpu().numpy())
		return np.vstack(all_features)

	def fit(self, X: np.ndarray, y: np.ndarray):
		features = self.extract_features(X)
		features = self.scaler.fit_transform(features)

		param_grid = {
			"n_neighbors": list(range(2, 15)),
			"weights": ["uniform", "distance"]
		}
		grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=3, scoring="accuracy", n_jobs=1)
		grid.fit(features, y)

		self.knn = grid.best_estimator_
		print("Лучший k:", self.knn.n_neighbors, "Вес:", self.knn.weights, "Точность:", grid.best_score_)

	def predict(self, X: np.ndarray) -> np.ndarray:
		"""
		Предсказание классов
		"""
		if self.knn is None:
			raise ValueError("Модель KNN не обучена. Сначала вызовите fit().")

		features = self.extract_features(X)
		features = self.scaler.transform(features)
		return self.knn.predict(features)

	def save(self, path: str='model.pkl'):
		"""
		Сохраняем KNN и StandardScaler
		"""
		if self.knn is None:
			raise ValueError("Нет обученной модели KNN для сохранения.")
		joblib.dump({"knn": self.knn, "scaler": self.scaler}, path)

	def load(self, path: str='model.pkl'):
		"""
		Загружаем KNN и StandardScaler
		"""
		checkpoint = joblib.load(path)
		self.knn = checkpoint["knn"]
		self.scaler = checkpoint["scaler"]


In [8]:
model = CNN_KNN_Model()
model.fit(images, labels)

Fitting 3 folds for each of 26 candidates, totalling 78 fits
Лучший k: 2 Вес: uniform Точность: 1.0


In [10]:
model.save('classification_model.pkl')

In [11]:
image = images[0]
model = CNN_KNN_Model('classification_model.pkl')
print(model.predict(image))

[1]
